# Part 2: Data Preprocessing
This notebook processes the raw Olist dataset files and prepares the product-level features for the Fuzzy Inference System, incorporating original min-max, log-transform, and rank-based normalization methods.

In [1]:
import pandas as pd
import numpy as np
import os

print('Pandas version:', pd.__version__)
print('Numpy version:', np.__version__)

Pandas version: 3.0.0
Numpy version: 2.4.2


## 1. Load Raw Datasets
We load the orders, order items, reviews, and products datasets from Olist.

In [2]:
orders = pd.read_csv('dataset/olist_orders_dataset.csv')
items = pd.read_csv('dataset/olist_order_items_dataset.csv')
reviews = pd.read_csv('dataset/olist_order_reviews_dataset.csv')
products = pd.read_csv('dataset/olist_products_dataset.csv')

print('Orders shape:', orders.shape)
print('Items shape:', items.shape)
print('Reviews shape:', reviews.shape)
print('Products shape:', products.shape)

Orders shape: (99441, 8)
Items shape: (112650, 7)
Reviews shape: (99224, 7)
Products shape: (32951, 9)


## 2. Filter Orders and Join Tables
We filter the orders to only include 'delivered' status transactions and join all four datasets together.

In [3]:
delivered_orders = orders[orders['order_status'] == 'delivered']
merged = items.merge(delivered_orders, on='order_id', how='inner')
merged = merged.merge(reviews, on='order_id', how='left')
merged = merged.merge(products, on='product_id', how='left')

print('Merged dataset shape:', merged.shape)

Merged dataset shape: (110840, 28)


## 3. Aggregate at Product Level
We aggregate the transaction-level data to get the key features for each product.

In [4]:
agg = merged.groupby('product_id').agg(
    total_sold=('order_item_id', 'count'),
    jumlah_penjualan=('order_id', 'nunique'),
    avg_rating=('review_score', 'mean'),
    total_revenue=('price', 'sum'),
    category=('product_category_name', 'first')
).reset_index()

print('Aggregated products count:', len(agg))
print(agg.head())

Aggregated products count: 32216
                         product_id  total_sold  jumlah_penjualan  avg_rating  \
0  00066f42aeeb9f3007548bb9d3f33c38           1                 1         5.0   
1  00088930e925c41fd95ebfe695fd2655           1                 1         4.0   
2  0009406fd7479715e4bef61dd91f2462           1                 1         1.0   
3  000b8f95fcb9e0096488278317764d19           2                 2         5.0   
4  000d9be29b5207b54e86aa1b1ac54872           1                 1         5.0   

   total_revenue               category  
0         101.65             perfumaria  
1         129.90             automotivo  
2         229.00        cama_mesa_banho  
3         117.80  utilidades_domesticas  
4         199.00     relogios_presentes  


## 4. Filter Data Density First
To ensure statistical validity of the rating scores, we filter out products that have less than 5 unique orders first, before applying any normalization scaling. This ensures our scales span the full range $[0, 100]$ for active products.

In [5]:
# Filter products with unique orders >= 5
filtered = agg[agg['jumlah_penjualan'] >= 5].copy()
print('Filtered products count (orders >= 5):', len(filtered))

Filtered products count (orders >= 5): 4109


## 5. Normalization and Transformations on Active Products
We calculate three different sets of variables for inventory (stock) and demand (sales) on the filtered active products:
1. **Original (Min-Max)**: Linear Min-Max scaling on total sold.
2. **Log-Transform**: Applies a logarithmic transform $\log(1 + x)$ to total sold before Min-Max scaling, to compress the extreme right-skewness.
3. **Rank-Based**: Computes the rank of total sold / unique order count and scales it to $[0, 100]$, providing a uniform distribution.

In [6]:
from scipy.stats import rankdata

# A. Original (Min-Max)
min_sold = filtered['total_sold'].min()
max_sold = filtered['total_sold'].max()
filtered['stok_level_orig'] = 100.0 - ((filtered['total_sold'] - min_sold) / (max_sold - min_sold) * 100.0)

min_jual = filtered['jumlah_penjualan'].min()
max_jual = filtered['jumlah_penjualan'].max()
filtered['jumlah_penjualan_orig'] = (filtered['jumlah_penjualan'] - min_jual) / (max_jual - min_jual) * 100.0

# B. Log-Transform Normalization
log_sold = np.log1p(filtered['total_sold'])
min_log = log_sold.min()
max_log = log_sold.max()
filtered['jumlah_penjualan_log'] = (log_sold - min_log) / (max_log - min_log) * 100.0
filtered['stok_level_log'] = 100.0 - filtered['jumlah_penjualan_log']

# C. Rank-Based Normalization
filtered['jumlah_penjualan_rank'] = rankdata(filtered['jumlah_penjualan']) / len(filtered) * 100.0
filtered['stok_level_rank'] = 100.0 - filtered['jumlah_penjualan_rank']

print('Normalization check (means):')
print(filtered[['stok_level_orig', 'stok_level_log', 'stok_level_rank']].mean())

Normalization check (means):
stok_level_orig    97.950792
stok_level_log     84.872939
stok_level_rank    49.987832
dtype: float64


## 6. Imputation of Missing Values
We fill missing category names with 'unknown'. For missing review ratings, we impute them using the median rating of their respective product category. If the category median is also missing, we use the global median.

In [7]:
filtered['category'] = filtered['category'].fillna('unknown')

global_median = filtered['avg_rating'].median()
category_medians = filtered.groupby('category')['avg_rating'].transform('median')
filtered['avg_rating'] = filtered['avg_rating'].fillna(category_medians).fillna(global_median)

print('Missing values count after imputation:')
print(filtered.isnull().sum())

Missing values count after imputation:
product_id               0
total_sold               0
jumlah_penjualan         0
avg_rating               0
total_revenue            0
category                 0
stok_level_orig          0
jumlah_penjualan_orig    0
jumlah_penjualan_log     0
stok_level_log           0
jumlah_penjualan_rank    0
stok_level_rank          0
dtype: int64


## 7. Export Cleaned Dataset
Finally, we save the cleaned product features to `dataset/product_features_clean.csv`.

In [8]:
os.makedirs('dataset', exist_ok=True)
filtered.to_csv('dataset/product_features_clean.csv', index=False)
print('Cleaned dataset saved successfully to dataset/product_features_clean.csv. Total rows:', len(filtered))

Cleaned dataset saved successfully to dataset/product_features_clean.csv. Total rows: 4109
